# 07 - Evaluación Comparativa: Baseline vs Proposed (Semantic Mapping)

**Objetivo:** Ejecutar la validación en el conjunto de test para ambos modelos (el entrenado con clases originales y el entrenado con clases agrupadas).
Extraer y comparar visualmente las métricas más esenciales para el paper:
1. mAP@50 y mAP@50-95 (Rendimiento general).
2. Matrices de Confusión (Para demostrar que el mapeo reduce la confusión entre clases similares).
3. Curvas Precision-Recall.

In [ ]:
# Importar librerías necesarias
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Configuración de rutas
# Modelos
path_model_baseline = '/nfs/workspace/sebastian.toro/yolov8s.pt'
path_model_proposed = '/nfs/workspace/sebastian.toro/runs/detect/epic_full_epoch7/weights/best.pt'

# Datasets (archivos YAML)
yaml_dataset_original = 'ruta/a/tu_dataset_original.yaml' 
yaml_dataset_mapped = '/nfs/workspace/sebastian.toro/EPIC-KITCHENS/Ego-Kitchen-YOLO/notebooks/data/epic_train_subset/dataset.yaml'

# Directorio base donde guardaremos los resultados de esta evaluación
EVAL_DIR = 'eval_results'

## 1. Ejecutar Evaluación (Validación en Test Set)

In [ ]:
print("--- EVALUANDO MODELO BASELINE ---")
model_base = YOLO(path_model_baseline)
# project y name definen dónde se guardan los gráficos: eval_results/baseline
metrics_base = model_base.val(data=yaml_dataset_original, split='test', project=EVAL_DIR, name='baseline', plots=True)

print("\n--- EVALUANDO MODELO PROPUESTO (MAPPED) ---")
model_prop = YOLO(path_model_proposed)
metrics_prop = model_prop.val(data=yaml_dataset_mapped, split='test', project=EVAL_DIR, name='proposed', plots=True)

## 2. Comparación Cuantitativa Rápida (mAP)

In [ ]:
# Extraer mAP50-95 global
map_base = metrics_base.box.map    # mAP@50-95 global
map50_base = metrics_base.box.map50  # mAP@50 global

map_prop = metrics_prop.box.map
map50_prop = metrics_prop.box.map50

print(f"📊 RESULTADOS GLOBALES:")
print(f"Baseline - mAP@50: {map50_base:.4f} | mAP@50-95: {map_base:.4f}")
print(f"Proposed - mAP@50: {map50_prop:.4f} | mAP@50-95: {map_prop:.4f}")

Mejora = (map_prop - map_base) / map_base * 100
print(f"Mejora en mAP@50-95: {Mejora:.2f}%")

## 3. Comparación Visual: Matrices de Confusión

In [ ]:
# Rutas a las imágenes generadas automáticamente por YOLO
conf_matrix_base = f"{EVAL_DIR}/baseline/confusion_matrix_normalized.png"
conf_matrix_prop = f"{EVAL_DIR}/proposed/confusion_matrix_normalized.png"

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

if os.path.exists(conf_matrix_base):
    img_base = mpimg.imread(conf_matrix_base)
    axes[0].imshow(img_base)
    axes[0].set_title('Baseline Model - Matriz de Confusión', fontsize=16)
    axes[0].axis('off')

if os.path.exists(conf_matrix_prop):
    img_prop = mpimg.imread(conf_matrix_prop)
    axes[1].imshow(img_prop)
    axes[1].set_title('Proposed Model (Mapped) - Matriz de Confusión', fontsize=16)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Comparación Visual: Curvas Precision-Recall (PR)

In [ ]:
# Rutas a las curvas PR generadas por YOLO
pr_curve_base = f"{EVAL_DIR}/baseline/PR_curve.png"
pr_curve_prop = f"{EVAL_DIR}/proposed/PR_curve.png"

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

if os.path.exists(pr_curve_base):
    img_pr_base = mpimg.imread(pr_curve_base)
    axes[0].imshow(img_pr_base)
    axes[0].set_title('Baseline Model - PR Curve', fontsize=16)
    axes[0].axis('off')

if os.path.exists(pr_curve_prop):
    img_pr_prop = mpimg.imread(pr_curve_prop)
    axes[1].imshow(img_pr_prop)
    axes[1].set_title('Proposed Model (Mapped) - PR Curve', fontsize=16)
    axes[1].axis('off')

plt.tight_layout()
plt.show()